# NB-6: Question-Count Scaling — 50 / 100 / 200 / 500Q Sweep

Tests CSAM (8B model) at increasing question counts on both HotPotQA and MuSiQue
to show that results are consistent and not cherry-picked from a small sample.

**Runs:**
- HotPotQA: 8B at 50Q, 100Q, 200Q, 500Q (seed 42)
- MuSiQue:  8B at 50Q, 100Q, 200Q, 500Q (seed 42)

**Note:** 500Q on MuSiQue exhausts the dev set (~500 questions total).
Expect each run to use ~500 tokens × N questions on 8B (14,400 RPD on Groq free).

**Output directory:** `results/nb6_qscaling/`  
**Time estimate:** ~5 min/100Q for 8B; ~40 min total for full sweep

**Just run all cells top-to-bottom. No interaction needed after cell 3.**

## Step 1 — Install dependencies & clone repo

In [ ]:
import os, subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'sentence-transformers', 'hnswlib', 'python-dotenv', 'groq', 'requests'], check=True)
print('Deps installed')

REPO = 'https://github.com/Lamaq-Mujpurwala/CSAM-IPD-HALH.git'
REPO_DIR = '/kaggle/working/CSAM-IPD-HALH' if os.path.exists('/kaggle') else '/content/CSAM-IPD-HALH'

if os.path.exists(REPO_DIR):
    r = subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', 'main'],
                       capture_output=True, text=True)
    print(r.stdout.strip() or 'Already up to date')
    if r.returncode != 0:
        print('[WARN] pull failed:', r.stderr[:200])
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO, REPO_DIR], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

commit = subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True, cwd=REPO_DIR)
print(f'Commit: {commit.stdout.strip()}')
print(f'Working dir: {os.getcwd()}')

## Step 2 — Set API keys

**Kaggle:** Secrets → `GROQ_API_KEY` (+ optionally `GROQ_API_KEY_2` … `GROQ_API_KEY_5`)  
**Colab:** Left sidebar key icon → same secrets

In [ ]:
import os

def _load_secret(name: str) -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v: return v
    except Exception: pass
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception: pass
    return os.environ.get(name, '')

env_lines = []
key = _load_secret('GROQ_API_KEY')
if not key:
    raise RuntimeError('GROQ_API_KEY not found — add it to Kaggle/Colab Secrets')
os.environ['GROQ_API_KEY'] = key
env_lines.append(f'GROQ_API_KEY={key}')
print('GROQ_API_KEY loaded')

for i in range(2, 10):
    k = _load_secret(f'GROQ_API_KEY_{i}')
    if k:
        os.environ[f'GROQ_API_KEY_{i}'] = k
        env_lines.append(f'GROQ_API_KEY_{i}={k}')
        print(f'GROQ_API_KEY_{i} loaded')

with open('.env', 'w') as f:
    f.write('\n'.join(env_lines) + '\n')
print(f'\n.env written with {len(env_lines)} key(s)')

## Step 3 — Configure

In [ ]:
import os

# ── EDIT THESE IF NEEDED ────────────────────────────────────────────────────
QUESTION_COUNTS = [50, 100, 200, 500]  # Q counts to sweep
PROVIDER        = 'groq'
MODEL           = 'llama-3.1-8b-instant'
SEED            = 42
CHECKPOINT_DIR  = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
# ─────────────────────────────────────────────────────────────────────────────

HOTPOT_DS  = os.path.join(REPO_DIR, 'csam_project', 'benchmarks', 'data', 'hotpotqa_dev.json')
MUSIQUE_DS = os.path.join(REPO_DIR, 'csam_project', 'benchmarks', 'data', 'musique_dev.jsonl')
OUT_DIR    = os.path.join(REPO_DIR, 'results', 'nb6_qscaling')
os.makedirs(OUT_DIR, exist_ok=True)

print(f'Model:       {MODEL}')
print(f'Seed:        {SEED}')
print(f'Output dir:  {OUT_DIR}')
print(f'Q counts:    {QUESTION_COUNTS}')
print(f'HotPotQA DS: {HOTPOT_DS} ({"OK" if os.path.exists(HOTPOT_DS) else "MISSING"})')
print(f'MuSiQue DS:  {MUSIQUE_DS} ({"OK" if os.path.exists(MUSIQUE_DS) else "MISSING"})')

total_q = sum(QUESTION_COUNTS) * 2  # hotpotqa + musique
print(f'\nTotal API calls approx: ~{total_q} (8B allows 14,400/day)')

## Step 4 — HotPotQA question-count sweep (50 / 100 / 200 / 500Q)

In [ ]:
import subprocess, sys, os
os.chdir(REPO_DIR)

hotpot_results = []

for n_q in QUESTION_COUNTS:
    cmd = [
        sys.executable, '-m', 'csam_project.benchmarks.benchmark_hotpotqa',
        '--provider', PROVIDER,
        '--model', MODEL,
        '--questions', str(n_q),
        '--dataset', HOTPOT_DS,
        '--seed', str(SEED),
        '--checkpoint-dir', CHECKPOINT_DIR,
        '--output-dir', OUT_DIR,
    ]
    print(f'\nHotPotQA: {n_q}Q ...')
    result = subprocess.run(cmd, capture_output=False, text=True)
    status = 'OK' if result.returncode == 0 else 'FAIL'
    print(f'[{status}] HotPotQA {n_q}Q')
    hotpot_results.append((n_q, status))

print(f'\nHotPotQA sweep done: {hotpot_results}')

## Step 5 — MuSiQue question-count sweep (50 / 100 / 200 / 500Q)

In [ ]:
import subprocess, sys, os
os.chdir(REPO_DIR)

musique_results = []

for n_q in QUESTION_COUNTS:
    cmd = [
        sys.executable, '-m', 'csam_project.benchmarks.benchmark_musique',
        '--provider', PROVIDER,
        '--model', MODEL,
        '--questions', str(n_q),
        '--dataset', MUSIQUE_DS,
        '--seed', str(SEED),
        '--checkpoint-dir', CHECKPOINT_DIR,
        '--output-dir', OUT_DIR,
    ]
    print(f'\nMuSiQue: {n_q}Q ...')
    result = subprocess.run(cmd, capture_output=False, text=True)
    status = 'OK' if result.returncode == 0 else 'FAIL'
    print(f'[{status}] MuSiQue {n_q}Q')
    musique_results.append((n_q, status))

print(f'\nMuSiQue sweep done: {musique_results}')

## Step 6 — Results table (F1 vs question count)

In [ ]:
import json, os, glob

hotpot_files = sorted(glob.glob(os.path.join(OUT_DIR, 'results_hotpotqa_*.json')))
musique_files = sorted(glob.glob(os.path.join(OUT_DIR, 'results_musique_*.json')))

safe_model = MODEL.replace('/', '_')

def extract_row(fp):
    with open(fp) as f: d = json.load(f)
    f1  = d.get('avg_f1', d.get('micro_f1', 0))
    sem = d.get('avg_semantic_sim', 0)
    em  = d.get('avg_em', d.get('exact_match', 0))
    n   = d.get('num_questions', 0)
    return f1, sem, em, n

print('=' * 65)
print('HOTPOTQA — F1 vs Question Count')
print('=' * 65)
print(f'{"N Questions":>12} {"Avg F1":>10} {"Sem Sim":>10} {"EM":>8}')
print('-' * 45)
for fp in hotpot_files:
    f1, sem, em, n = extract_row(fp)
    print(f'{n:>12} {f1:>10.4f} {sem:>10.4f} {em:>8.4f}')

print('\n' + '=' * 65)
print('MUSIQUE — F1 vs Question Count')
print('=' * 65)
print(f'{"N Questions":>12} {"Avg F1":>10} {"Sem Sim":>10} {"EM":>8}')
print('-' * 45)
for fp in musique_files:
    f1, sem, em, n = extract_row(fp)
    print(f'{n:>12} {f1:>10.4f} {sem:>10.4f} {em:>8.4f}')

print(f'\nTotal files: {len(hotpot_files)} HotPotQA + {len(musique_files)} MuSiQue')

## Step 7 — Save / Download results

In [ ]:
import shutil, os, glob

all_files = glob.glob(os.path.join(OUT_DIR, '*.json'))

if os.path.exists('/kaggle'):
    kaggle_out = '/kaggle/working/nb6_qscaling'
    os.makedirs(kaggle_out, exist_ok=True)
    for fp in all_files:
        dest = os.path.join(kaggle_out, os.path.basename(fp))
        shutil.copy(fp, dest)
        print(f'Kaggle output: {dest}')
else:
    try:
        from google.colab import files
        for fp in all_files:
            files.download(fp)
            print(f'Downloaded: {fp}')
    except ImportError:
        print('Files saved at:')
        for fp in sorted(all_files): print(f'  {fp}')